# 02 - Nsight Compute Kernel Analysis

Notebook 01 used Nsight Systems to answer **where** time went in the training pipeline. This optional notebook uses Nsight Compute to ask a narrower question: when the remaining work is inside GPU kernels, **why** do selected kernels behave the way they do?

The goal is not to rewrite PyTorch or cuBLAS kernels. The goal is to learn when NCU points toward a better high-level formulation, and when it tells you that the hot kernel is already a strong library implementation.


## What We Will Profile

We focus on the classifier head because the previous notebooks already made the CPU/GPU handoffs, synchronization, and batch IO easier to reason about. The two heads compute the same squared-distance scores in different ways:

- `broadcast-distance`: expands samples and prototypes into a larger intermediate tensor, then uses elementwise kernels and reductions.
- `matmul-distance`: rewrites the same distance calculation around a matrix multiply, which lets PyTorch route the dominant work into library kernels.

In a real project, this is a common NCU lesson: the fix is often changing tensor math so the framework can choose better kernels, not manually optimizing a vendor kernel.


## Setup

The runtime cells work on CPU for quick exploration. Nsight Compute capture requires CUDA and `ncu` on `PATH`; when either is missing, the notebook prints the commands it would run and skips report capture.


In [ ]:
from __future__ import annotations

from pathlib import Path
import os
import re
import shutil
import subprocess
import sys
from typing import Sequence

from IPython.display import Markdown, display
import torch

ROOT = Path.cwd()
if not (ROOT / "profiling_workshop").exists() and (ROOT.parent / "profiling_workshop").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from profiling_workshop.common import cuda_is_available_quietly

TRACE_DIR = ROOT / "traces"
TRACE_DIR.mkdir(exist_ok=True)
HEAD_SCRIPT = ROOT / "scripts" / "profile_classifier_head.py"

PROFILE_DEVICE = "cuda" if cuda_is_available_quietly() else "cpu"
NCU = shutil.which("ncu")
NCU_UI = shutil.which("ncu-ui")
RUN_NCU = PROFILE_DEVICE == "cuda" and NCU is not None

print(f"Workshop root: {ROOT}")
print(f"Trace directory: {TRACE_DIR}")
print(f"Profile device: {PROFILE_DEVICE}")
print(f"ncu: {NCU or 'not found'}")
print(f"ncu-ui: {NCU_UI or 'not found'}")
if PROFILE_DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")


## Workload Arguments

The CUDA defaults are sized so the classifier head launches enough work to inspect on L40S/A100-class GPUs without profiling the whole training loop. Increase `--iterations` if NCU captures too few matching launches, or lower `--batch-size`/`--hidden` for smaller GPUs.


In [ ]:
if PROFILE_DEVICE == "cuda":
    BASE_ARGS = [
        "--device", "cuda",
        "--batch-size", "1024",
        "--hidden", "4096",
        "--classes", "64",
        "--iterations", "20",
        "--warmup", "5",
    ]
else:
    BASE_ARGS = [
        "--device", "cpu",
        "--batch-size", "16",
        "--hidden", "64",
        "--classes", "8",
        "--iterations", "2",
        "--warmup", "1",
    ]

BROADCAST_ARGS = [*BASE_ARGS, "--head", "broadcast-distance"]
MATMUL_ARGS = [*BASE_ARGS, "--head", "matmul-distance"]
MATMUL_AMP_ARGS = [*MATMUL_ARGS, "--amp"]

BROADCAST_ARGS, MATMUL_ARGS, MATMUL_AMP_ARGS


## Helpers

The NCU helper uses NVTX range filtering so the profiler only targets kernels launched inside `ncu_classifier_head_*`. It also limits collection with `--launch-count`; NCU may replay selected kernels multiple times to collect metrics, so profiling too many launches gets slow quickly.


In [ ]:
RESULT_RE = re.compile(
    r"RESULT head=(?P<head>\S+) device=(?P<device>\S+) dtype=(?P<dtype>\S+) "
    r"iterations=(?P<iterations>\d+) samples=(?P<samples>\d+) seconds=(?P<seconds>[0-9.]+) "
    r"samples_per_second=(?P<sps>[0-9.]+) checksum=(?P<checksum>-?[0-9.]+)"
)


def run_command(cmd: Sequence[str], check: bool = True, capture: bool = False) -> subprocess.CompletedProcess[str]:
    cmd = [str(part) for part in cmd]
    print("\n$ " + " ".join(cmd))
    return subprocess.run(cmd, check=check, text=True, capture_output=capture)


def parse_head_output(output: str) -> dict[str, object]:
    match = RESULT_RE.search(output)
    if match is None:
        raise ValueError("The profiling target did not print a RESULT line.")
    data = match.groupdict()
    return {
        "head": data["head"],
        "device": data["device"],
        "dtype": data["dtype"],
        "iterations": int(data["iterations"]),
        "samples": int(data["samples"]),
        "seconds": float(data["seconds"]),
        "samples_per_second": float(data["sps"]),
        "checksum": float(data["checksum"]),
        "output": output,
    }


def run_head_target(label: str, args: Sequence[str]) -> dict[str, object]:
    completed = run_command([sys.executable, HEAD_SCRIPT, *args], check=True, capture=True)
    print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    result = parse_head_output(completed.stdout)
    result["label"] = label
    return result


def comparison_table(*runs: dict[str, object]) -> None:
    rows = [
        "| label | head | dtype | seconds | samples/s | checksum |",
        "|---|---|---|---:|---:|---:|",
    ]
    for run in runs:
        rows.append(
            f"| {run['label']} | {run['head']} | {run['dtype']} | "
            f"{run['seconds']:.6f} | {run['samples_per_second']:,.1f} | {run['checksum']:.6f} |"
        )
    display(Markdown("\n".join(rows)))


def remove_stale_ncu_report(report_base: Path) -> None:
    report = report_base.with_suffix(".ncu-rep")
    if report.exists():
        report.unlink()


def ncu_profile(
    name: str,
    nvtx_range: str,
    script_args: Sequence[str],
    *,
    launch_count: int = 10,
) -> Path | None:
    report_base = TRACE_DIR / name
    report = report_base.with_suffix(".ncu-rep")
    cmd = [
        "ncu",
        "--force-overwrite",
        "--target-processes", "all",
        "--set", os.environ.get("NCU_SET", "full"),
        "--launch-count", str(launch_count),
        "--nvtx",
        "--nvtx-include", f"{nvtx_range}/",
        "-o", str(report_base),
        sys.executable,
        str(HEAD_SCRIPT),
        *script_args,
    ]
    if not RUN_NCU:
        print("Skipping Nsight Compute capture because CUDA or ncu is unavailable.")
        print("Would run:")
        print(" ".join(str(part) for part in cmd))
        return None

    remove_stale_ncu_report(report_base)
    completed = run_command(cmd, check=False, capture=False)
    if report.exists():
        print(f"Nsight Compute report: {report}")
        return report
    if completed.returncode != 0:
        print(f"ncu exited with return code {completed.returncode}")
    return None


def ncu_import_details(report: Path | None) -> None:
    if report is None:
        print("No Nsight Compute report to summarize.")
        return
    if shutil.which("ncu") is None:
        print("ncu was not found on PATH.")
        return
    run_command(["ncu", "--import", report, "--page", "details"], check=False, capture=False)


## Quick Runtime Comparison

Before collecting detailed kernel metrics, run the narrow target directly. Treat this as a quick sanity check, not a substitute for NCU. The checksums are just there to keep the forward pass alive and make it harder to accidentally benchmark nothing.


In [ ]:
broadcast_runtime = run_head_target("broadcast", BROADCAST_ARGS)
matmul_runtime = run_head_target("matmul", MATMUL_ARGS)
comparison_table(broadcast_runtime, matmul_runtime)


## Experiment 1: Broadcast Distance Head

Open the generated report in `ncu-ui` and inspect Speed of Light, Roofline, Memory Workload, Launch Statistics, and Occupancy. The expected story is a less tidy collection of elementwise/reduction-style kernels and more memory movement from the broadcasted intermediate.


In [ ]:
broadcast_report = ncu_profile(
    "ncu_broadcast_head",
    "ncu_classifier_head_broadcast_distance",
    BROADCAST_ARGS,
    launch_count=10,
)
broadcast_report


In [ ]:
ncu_import_details(broadcast_report)


## Experiment 2: Matmul Distance Head

The matmul formulation computes the same squared-distance scores, but it changes the operation mix. In NCU, look for the dominant work moving into GEMM/library kernels and ask whether those kernels are compute-bound, memory-bound, or already near a sensible roofline for the selected shape.


In [ ]:
matmul_report = ncu_profile(
    "ncu_matmul_head",
    "ncu_classifier_head_matmul_distance",
    MATMUL_ARGS,
    launch_count=10,
)
matmul_report


In [ ]:
ncu_import_details(matmul_report)


## Experiment 3: Optional AMP Comparison

On supported GPUs, AMP may route suitable matrix operations through Tensor Core paths. Use this as a discussion of Tensor Core utilization and dtype tradeoffs, not as a promise that AMP always wins.


In [ ]:
if PROFILE_DEVICE == "cuda":
    matmul_amp_runtime = run_head_target("matmul amp", MATMUL_AMP_ARGS)
    comparison_table(matmul_runtime, matmul_amp_runtime)
else:
    print("Skipping AMP runtime comparison because CUDA is unavailable.")


In [ ]:
matmul_amp_report = ncu_profile(
    "ncu_matmul_head_amp",
    "ncu_classifier_head_matmul_distance",
    MATMUL_AMP_ARGS,
    launch_count=10,
)
matmul_amp_report


## Close the Loop

The takeaway is deliberately modest: NCU is excellent for understanding selected kernels, but PyTorch users often act on that information by changing tensor shapes, batching, dtype, or operation formulation. If NCU shows that the remaining hot kernel is a strong library GEMM, the right optimization may be to stop kernel tuning and return to model or pipeline decisions.

Official references:

- NVIDIA Nsight Compute CLI: https://docs.nvidia.com/nsight-compute/NsightComputeCli/index.html
- NVIDIA Nsight Compute Profiling Guide: https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html
